# Module 3 · Virtual Biotech: from team to evidence

**Swarm in the Lab** · BiocAsia 2026 · 15:40–16:30

Module 1's agents argued from memory. Module 2 gave agents tools. Here a whole organisation gets both. A **Chief Scientific Officer (CSO)** agent delegates your research strategy to specialist agents. They query real data (Open Targets, CELLxGENE, ClinicalTrials.gov) and your Module 2 paper agent, and every claim in the final report has to point to the file or tool call behind it.

| Time | Section | What you do |
|---|---|---|
| 15:40 | **1 · Walkthrough** | The organisation chart, how a question is delegated, the MCP tool servers, and a **dry-run routing trace** that costs nothing |
| 15:55 | **2 · Hands-on** | Assemble a small team, attach your paper agent, send it the Module 1 strategy and watch it work |
| 16:20 | **3 · Read the report** | Target rationale, safety liabilities, suggested modality, and the source behind each claim |
| — | **⭐ Stretch** | Add a new division agent and see how routing changes |

**What you leave with:** a run folder containing the report, the claims and their evidence, and a full trace. `run_summary.json` points Module 4 at it.

**Running out of time, or the run fails?** Set `USE_CACHE = True` in the next cell and continue from Section 3 with a pre-run copy.

> This module runs the published Virtual Biotech code unchanged. A small workshop launcher (`tools/vbt_workshop.py`) chooses the team, adds your paper agent and sets a time limit. It also adds guards that block the obvious ways an agent's shell command could print your key. They reduce that risk but can't remove it, which is why workshop keys have spending caps and are revoked afterwards. Section 3 checks whether your key appears anywhere in the run's records.
>
> To stop a run early, interrupt the kernel (the ⏹ button). The run is stopped and saved, rather than left running on your key.

## 0 · Setup

In [ ]:
from pathlib import Path

WORKSHOP_DIR      = Path.home() / "workshop"
VBT_DIR           = WORKSHOP_DIR / "virtual-biotech"                  # the TheVirtualBiotech checkout
CONDA_BASE        = Path.home() / "miniforge3"
VBT_ENV           = "vbt"                                             # its conda environment (Python 3.11)
OPEN_TARGETS_DATA = WORKSHOP_DIR / "data" / "open_targets"            # Open Targets 25.09 download
LAUNCHER          = WORKSHOP_DIR / "tools" / "vbt_workshop.py"
OUTPUT_DIR        = WORKSHOP_DIR / "outputs" / "module3"
CACHE_DIR         = WORKSHOP_DIR / "cache" / "module3"
MODULE1_DIR       = WORKSHOP_DIR / "outputs" / "module1"
MODULE2_DIR       = WORKSHOP_DIR / "outputs" / "module2"

GENE, DISEASE  = "B7-H3 (CD276)", "lung cancer"
MODEL          = "claude-sonnet-5-5"   # CSO and specialists. The chief of staff and reviewer always use Haiku.
EFFORT         = "medium"              # specialists' reasoning effort (the paper used high/max: slower and costlier)
CSO_EFFORT     = "high"                # the paper used xhigh
TIME_LIMIT_MIN = 15                    # the team's research is stopped after this long …
WRAPUP_MIN     = 4                     # … then the CSO gets this long to write up what is on record
EXPECTED_COST  = "about US$4–7 (measured on the organisers' test runs)"
USE_CACHE      = False                 # True = read a pre-run copy instead of running the team

# Your Anthropic key is read from the private key file (self-test notebook, Step 0) and passed only to the
# Virtual Biotech process. Never paste a key into this notebook.

In [ ]:
# Imports and helper functions. Skim the comments; you don't need to read every line.
import collections, datetime, hashlib, html, json, os, queue, re, shutil, signal, subprocess, threading, time
from IPython.display import display, Markdown, HTML

# --- Hiding secrets ------------------------------------------------------------------
# Anything that looks like a key or token is replaced before it's shown on screen or saved.
_SECRET_VALUES = set()
_SECRET_RE = re.compile(   # (?<!…) stops ordinary words like "risk-assessment" from matching
    r"(?<![A-Za-z0-9])(?:sk-[A-Za-z0-9_*.\-]{16,}|hf_[A-Za-z0-9]{20,}|gh[pousr]_[A-Za-z0-9]{20,}"
    r"|github_pat_[A-Za-z0-9_]{20,}|(?:AKIA|ASIA)[A-Z0-9]{16}\b"
    r"|eyJ[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,}\.[A-Za-z0-9_\-]{10,})")

def redact(text):
    s = "" if text is None else str(text)
    for value in _SECRET_VALUES:
        s = s.replace(value, "[hidden]")
    return _SECRET_RE.sub("[hidden]", s)

# --- Workshop keys --------------------------------------------------------------------
# Keys live in one file that only your user can read. They are loaded into this notebook's
# process only: not into your terminal, and not into Claude Code. They are never printed.
KEY_FILE = Path.home() / ".config" / "swarm-lab" / "keys.env"
KEY_NAMES = ("OPENAI_API_KEY", "ANTHROPIC_API_KEY", "OPENAI_BASE_URL", "ANTHROPIC_BASE_URL")

def _read_key_file():
    values = {}
    try:
        lines = KEY_FILE.read_text().splitlines()
    except OSError:
        return values
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        name = name.strip().removeprefix("export ").strip()
        value = value.strip().strip('"').strip("'")
        if name in KEY_NAMES and value:
            values[name] = value
    return values

def key_file_is_private():
    """True if only the owner can read the key file and its folder."""
    try:
        return (KEY_FILE.stat().st_mode & 0o077) == 0 and (KEY_FILE.parent.stat().st_mode & 0o077) == 0
    except OSError:
        return False

def load_keys(names):
    """Load the named keys into this notebook only. Returns where each came from:
    'environment', 'key file' or None (missing)."""
    from_file = _read_key_file()
    sources = {}
    for name in names:
        if os.environ.get(name):
            sources[name] = "environment"
        elif name in from_file:
            os.environ[name] = from_file[name]
            sources[name] = "key file"
        else:
            sources[name] = None
        if name.endswith("_KEY") and os.environ.get(name):
            _SECRET_VALUES.add(os.environ[name])
    return sources

def fingerprint(value):
    """A short label that tells keys apart without revealing them."""
    return hashlib.sha256(value.encode()).hexdigest()[:8]

load_keys(["ANTHROPIC_API_KEY", "ANTHROPIC_BASE_URL"])   # this module needs only the Anthropic key
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _esc(x):
    return html.escape(redact(x))

def _read_json(path):
    try:
        return json.loads(Path(path).read_text())
    except (OSError, ValueError):
        return None

def table(headers, rows, title=None):
    head = "".join(f"<th style='text-align:left'>{_esc(h)}</th>" for h in headers)
    body = "".join("<tr>" + "".join(f"<td style='text-align:left;vertical-align:top'>{_esc(c)}</td>" for c in r) + "</tr>" for r in rows)
    display(HTML((f"<p><b>{_esc(title)}</b></p>" if title else "") + f"<table><tr>{head}</tr>{body}</table>"))

def show_text(title, text, open_=False):
    display(HTML(f"<details{' open' if open_ else ''}><summary><b>{_esc(title)}</b></summary>"
                 f"<div style='white-space:pre-wrap;font-size:0.88em;margin:6px 0 0 12px'>{_esc(text)}</div></details>"))


# --- Running things in the Virtual Biotech's own environment (Python 3.11) --------------
def _command(args):
    conda_sh = CONDA_BASE / "etc" / "profile.d" / "conda.sh"
    if conda_sh.exists():   # activate properly, so R and other tools in the environment are found
        return ["bash", "-c", 'source "$1" && conda activate "$2" && shift 2 && exec python -u "$@"',
                "vbt", str(conda_sh), VBT_ENV, *map(str, args)]
    return [str(CONDA_BASE / "envs" / VBT_ENV / "bin" / "python"), "-u", *map(str, args)]

def _child_env(with_key):
    env = {k: v for k, v in os.environ.items() if not k.startswith(("OPENAI_", "ANTHROPIC_"))}
    env.update(PYTHONUNBUFFERED="1", OPEN_TARGETS_DATA_PATH=str(OPEN_TARGETS_DATA))
    if with_key:
        for name in ("ANTHROPIC_API_KEY", "ANTHROPIC_BASE_URL"):
            if os.environ.get(name):
                env[name] = os.environ[name]
    return env

def launch(args, with_key=False, timeout_s=600, on_event=None):
    """Run the launcher. Returns (events, exit code, last log lines). Lines starting with § are events;
    everything else goes to launcher.log. All output is passed through redact() first."""
    try:
        proc = subprocess.Popen(_command(args), cwd=str(OUTPUT_DIR), env=_child_env(with_key), text=True, bufsize=1,
                                stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                start_new_session=True)
    except OSError as e:
        print(f"❌ Couldn't start the Virtual Biotech's Python ({e.strerror}: {e.filename}). "
              f"Organisers: check CONDA_BASE and VBT_ENV in the Setup cell. Otherwise, tell a helper.")
        return [], 127, []
    lines = queue.Queue()
    def reader():
        for line in proc.stdout:
            lines.put(line)
        lines.put(None)
    threading.Thread(target=reader, daemon=True).start()
    def stop(hard):
        # Gentle: ask the launcher to stop the team and save the run. Hard: end the launcher and everything it started.
        try:
            if hard:
                os.killpg(proc.pid, signal.SIGKILL)
            else:
                os.kill(proc.pid, signal.SIGINT)
        except ProcessLookupError:
            pass

    events, tail, deadline, stage = [], collections.deque(maxlen=40), time.time() + timeout_s, 0
    with open(OUTPUT_DIR / "launcher.log", "a") as log:
        while True:
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                if time.time() > deadline and proc.poll() is None:
                    stage += 1
                    print("⏱️ Taking too long: asking the run to stop and save …" if stage == 1 else "⏱️ Forcing it to stop.")
                    stop(hard=stage > 1)
                    deadline = time.time() + 150
                continue
            except KeyboardInterrupt:
                # You pressed Stop (interrupt the kernel). Don't leave the team running on your key.
                stage += 1
                print("⏹️ Stopping the run. It saves what it has done, which takes up to two minutes. "
                      "Interrupt again to force it." if stage == 1 else "⏹️ Forcing it to stop.")
                stop(hard=stage > 1)
                deadline = time.time() + 150
                continue
            if line is None:
                break
            line = redact(line.rstrip("\n"))
            if line.startswith("§"):
                try:
                    event = json.loads(line[1:])
                except ValueError:
                    log.write(line + "\n")
                    continue
                events.append(event)
                if on_event:
                    on_event(event)
            else:
                log.write(line + "\n")
                tail.append(line)
    proc.wait()
    return events, proc.returncode, list(tail)

def show_event(e):
    k = e.get("kind")
    if k == "status":
        print("•", e.get("message"))
    elif k == "run_dir":
        print("📁 Run folder:", e.get("path"))
    elif k == "progress":
        s = e.get("elapsed_s", 0)
        failed = f" ({e['failed_calls']} failed)" if e.get("failed_calls") else ""
        phase = " · writing up" if e.get("phase") == "wrap-up" else ""
        print(f"  {s // 60:>2}:{s % 60:02d}{phase} · working: {', '.join(e.get('running') or []) or '—'} · "
              f"finished: {', '.join(e.get('finished') or []) or '—'} · {e.get('tool_calls', 0)} tool calls{failed}")
    elif k in ("time_limit", "stopping"):
        print("⏱️" if k == "time_limit" else "⏹️", e.get("message"))
    elif k == "wrapup":
        print("📝", e.get("message"))
    elif k == "error":
        print("❌", e.get("message"))


def run_cost(run_dir):
    """(low, high) in US$. The run records two figures: its own total (from the CSO's turns) and what each
    specialist reported when it finished. Specialists that run in the background aren't always counted in the
    run's total, so the true cost lies between the larger of the two and their sum. Agents that never finished
    report nothing, so a run that was cut short may cost more than either."""
    run_dir = Path(run_dir)
    report = _read_json(run_dir / "logs" / "cost_report.json") or {}
    total = report.get("total_cost_usd") or 0
    agents = 0.0
    trace = run_dir / "logs" / "trace.jsonl"
    if trace.exists():
        for line in trace.read_text().splitlines():
            try:
                ev = json.loads(line)
            except ValueError:
                continue
            c = (ev.get("cost") or {}) if ev.get("type") == "agent_stop" else {}
            amount = c.get("total_cost_usd") or c.get("cost_usd")
            if isinstance(amount, (int, float)):
                agents += amount
    return max(total, agents), total + agents

def cost_text(run_dir):
    low, high = run_cost(run_dir)
    if not high:
        return "not recorded"
    return f"US${low:.2f}" if abs(high - low) < 0.01 else f"between US${low:.2f} and US${high:.2f}"


# --- The team configuration the launcher reads ----------------------------------------------
def paper_agents():
    for p in (MODULE2_DIR / "paper_agents.json", WORKSHOP_DIR / "cache" / "module2" / "paper_agents.json"):
        data = _read_json(p)
        if data:
            return data
    return {}

def write_config(team, custom_agents=None, paper_agent_holder=None):
    pa = paper_agents()
    cfg = {
        "vbt_dir": str(VBT_DIR), "runs_dir": str(OUTPUT_DIR / "runs"),
        "team": list(team), "custom_agents": custom_agents or {},
        "model": MODEL, "effort": EFFORT, "cso_effort": CSO_EFFORT,
        "time_limit_min": TIME_LIMIT_MIN, "wrapup_min": WRAPUP_MIN, "specialist_max_turns": 40, "cso_max_turns": 60,
        "paper_agents": {"servers": pa.get("servers", {}), "tools": pa.get("tools", {}),
                         "dataset": pa.get("dataset"), "attach_to": paper_agent_holder} if pa else {},
    }
    path = OUTPUT_DIR / "team_config.json"
    path.write_text(json.dumps(cfg, indent=2))
    return path

def vbt_plan(team, custom_agents=None, paper_agent_holder=None):
    """The dry run: the team the CSO would get, and each agent's tools. No model calls, no cost."""
    cfg = write_config(team, custom_agents, paper_agent_holder)
    events, code, tail = launch([LAUNCHER, "plan", "--config", cfg], timeout_s=120)
    plan = next((e for e in events if e.get("kind") == "plan"), None)
    if plan is None:
        print("❌ The dry run failed. Last lines of the log:\n  " + "\n  ".join(tail[-12:]))
    return plan


# --- Routing preview: which specialist would most likely take each analysis ---------------
STRATEGY_SERVERS = {   # Module 1's tool names -> the MCP servers (or built-in tools) that can do that work
    "open_targets": ["target", "association", "genetics", "disease", "drug"],
    "cellxgene_census": ["single_cell", "expression"],
    "clinicaltrials_gov": ["clinicaltrials"],
    "chembl": ["drug"],
    "pubmed": ["WebSearch"],
}

def likely_agents(tool, plan):
    wanted = STRATEGY_SERVERS.get(str(tool).lower(), [str(tool).lower()])   # paper agents: server name = tool name
    scored = []
    for a in plan["team"]:
        n = sum(len(a["tools_by_server"].get(s, [])) for s in wanted)
        n += 1 if "WebSearch" in wanted and "WebSearch" in a["builtin_tools"] else 0
        if n:
            scored.append((n, a["name"]))
    return [name for _, name in sorted(scored, reverse=True)]

def routing_preview(plan, analyses):
    names = {a["name"] for a in plan["team"]}
    rows, routed = [], set()
    for a in analyses:
        who = likely_agents(a.get("tool"), plan)
        routed.update(who[:1])
        rows.append([a.get("id"), a.get("tool"), a.get("question"),
                     " → ".join(who[:2]) if who else "⚠️ nobody on this team has these tools"])
    table(["ID", "Tool", "Analysis", "Most likely specialist"], rows, title="Dry-run routing trace: who would do what")
    specialists = sorted(routed - {"chief-of-staff", "scientific-reviewer"})
    print("Expected order of work (a preview from tool ownership; the CSO makes the real decisions):")
    print("  1 · CSO reads the brief")
    print(f"  2 · chief-of-staff writes a field briefing from the web   {'✅' if 'chief-of-staff' in names else '— not on this team'}")
    print("  3 · CSO records an analysis plan (provenance.write_plan)")
    print(f"  4 · specialists work in parallel, each in its own folder: {', '.join(specialists) or '—'}")
    print(f"  5 · scientific-reviewer checks their outputs (read-only)   {'✅' if 'scientific-reviewer' in names else '— not on this team'}")
    print("  6 · CSO writes the answer and files claims; each must point to a file or tool call on record")
    return routed | (names & {"chief-of-staff", "scientific-reviewer"})

def section(text, keyword):
    """The part of a Markdown report under the first heading that contains `keyword`."""
    lines, out, level = (text or "").splitlines(), None, None
    for line in lines:
        m = re.match(r"^(#{1,6})\s+(.*)", line)
        if m:
            if out is not None and len(m.group(1)) <= level:
                break
            if out is None and keyword.lower() in m.group(2).lower():
                out, level = [], len(m.group(1))
                continue
        if out is not None:
            out.append(line)
    return "\n".join(out).strip() if out else None

print(f"Virtual Biotech at {VBT_DIR} · model {MODEL} · effort {EFFORT}"
      + (f" · Anthropic key fingerprint {fingerprint(os.environ['ANTHROPIC_API_KEY'])}" if os.environ.get("ANTHROPIC_API_KEY") else
         " · ⚠️ no Anthropic key (run Step 0 of the self-test, or use USE_CACHE = True)")
      + (" · USING PRE-RUN COPY" if USE_CACHE else ""))

## 1 · Walkthrough: how the Virtual Biotech works · 15:40–15:55

### 1a · The organisation chart
The CSO never touches data. It plans, delegates and synthesises. The **chief of staff** writes a quick field briefing from the web, the **scientific reviewer** checks specialists' work (read-only), and the scientists sit in four divisions. The table below comes from the real code: each agent's **description** is all the CSO reads when it decides whom to call.

In [ ]:
DEFAULT_TEAM = ["chief-of-staff", "single-cell-analyst", "fda-safety-officer", "clinical-trialist",
                "medchem-pharmacologist", "scientific-reviewer"]
PLAN = vbt_plan(DEFAULT_TEAM, paper_agent_holder="single-cell-analyst")
if PLAN:
    table(["Agent", "Division and job (what the CSO reads)"],
          [[a["name"], a["description"]] for a in PLAN["available"]], title=f"{len(PLAN['available'])} agents in the full organisation")

### 1b · How a question flows
1. **The CSO reads the question**, plus its *run-organisation* skill, which tells it how to keep the run auditable.
2. **The chief of staff** briefs it on the field and on what data exists.
3. **The CSO writes a plan** as a small dependency graph (`inputs/plan.json`). After the run, the plan is compared with what actually happened.
4. **Specialists run in parallel.** Each is a separate Claude agent with its own system prompt and its own tool list. Each works only inside its own folder (`work/<agent>/`) and registers the files it produces.
5. **The scientific reviewer** reads their outputs and checks the methods, the evidence, and whether they answered the question.
6. **The CSO writes the answer and files claims** (`evidence/claims.json`). A claim is accepted only if its evidence points at a file in the run's manifest, or a tool call in its trace. A citation that can't be checked locally is marked *external*.

Everything is recorded in `logs/trace.jsonl`, and `audit.html` lets you browse it. The agents also run inside a sandbox: they can't read outside their folders, install packages, or run destructive commands.

### 1c · The tool servers
Specialists reach data through MCP servers, the same protocol as Module 2's paper agents. Most read a local copy of Open Targets (release 25.09) instead of calling a web API, so results are reproducible and nothing is rate-limited.

In [ ]:
SERVER_SOURCES = {
    "expression": "GTEx bulk RNA-seq", "genetics": "Open Targets Genetics (GWAS, L2G, QTL)",
    "target": "Open Targets target annotations, tractability, safety", "drug": "Open Targets drugs (ChEMBL-derived)",
    "disease": "Open Targets disease ontology", "association": "Open Targets target–disease associations",
    "single_cell": "CELLxGENE Census", "functional_genomics": "DepMap CRISPR (+ Tahoe, optional)",
    "pathway": "Reactome + Gene Ontology", "interaction": "protein–protein interactions",
    "clinicaltrials": "ClinicalTrials.gov + cBioPortal", "provenance": "the run's own plan, artifacts and claims",
}
if PLAN:
    users = collections.defaultdict(list)
    for a in PLAN["team"]:
        for server in a["tools_by_server"]:
            users[server].append(a["name"])
    table(["Server", "Data", "Used by (on the default team)", "Ready?"],
          [[s["name"], "paper agent (Module 2)" if s["paper_agent"] else SERVER_SOURCES.get(s["name"], ""),
            ", ".join(users.get(s["name"], ["CSO"])), "✅" if s["configured"] else "❌ not configured"]
           for s in PLAN["servers"]], title="Servers this team would start")
    if not PLAN.get("mcp_config_found"):
        print("❌ mcp_config.json is missing. Organisers: run `python setup_mcp.py` in the Virtual Biotech folder.")

### 💬 Ask Claude Code
Paste one of these into Claude Code, replacing `<vbt>` with the folder printed by the next cell. Claude Code may read files there, but please don't ask it to run the Virtual Biotech itself: that's what Section 2 is for.

1. *"Read `<vbt>/src/agents/registry.py`. Which tools does the single-cell-analyst have that the fda-safety-officer doesn't, and why might that split be sensible?"*
2. *"In `<vbt>/src/agents/cso/system_prompt.md`, when is the CSO told to ask clarifying questions, and when to call the chief of staff?"*
3. *"Read `<vbt>/src/utils/claims.py`. What happens when the CSO files a claim citing a file that doesn't exist?"*
4. *"Read `<vbt>/src/utils/agent_hooks.py`. Which shell commands are agents blocked from running, and what can still get through?"*

In [ ]:
print(VBT_DIR)

### 1d · Dry-run routing trace
Before spending anything, look at which specialist would most likely take each analysis from your Module 1 strategy. The preview is based on who holds the right tools. It uses no model calls; the real routing is the CSO's decision, and Section 3 compares the two.

In [ ]:
strategy = _read_json(MODULE1_DIR / "research_strategy.json") or _read_json(WORKSHOP_DIR / "cache" / "module1" / "research_strategy.json") or {}
ANALYSES = strategy.get("analyses", [])
if not ANALYSES:
    print("No Module 1 strategy found; using two example analyses.")
    ANALYSES = [{"id": "A1", "tool": "cellxgene_census", "question": f"Expression of {GENE} across healthy tissues"},
                {"id": "A2", "tool": "clinicaltrials_gov", "question": f"Clinical trials of {GENE}-targeted therapies in {DISEASE}"}]
if PLAN:
    PREDICTED = routing_preview(PLAN, ANALYSES)

## 2 · Hands-on: assemble the team and send the strategy · 15:55–16:20

### 2a · Your team
The default is a scaled-down organisation: the CSO plus a chief of staff, four specialists and a reviewer. The specialists are single-cell (CELLxGENE), FDA safety (Open Targets safety data), clinical trials (ClinicalTrials.gov) and medicinal chemistry (drugs and modality). Your Module 2 paper agent is attached to the single-cell analyst.

🔵 Swap a specialist (e.g. `genomics-analyst` or `target-biologist`), then re-run this cell and the routing trace below. Fewer agents is faster and cheaper; every agent you add runs in parallel and costs money.

In [ ]:
TEAM = ["chief-of-staff", "single-cell-analyst", "fda-safety-officer", "clinical-trialist",
        "medchem-pharmacologist", "scientific-reviewer"]
PAPER_AGENT_HOLDER = "single-cell-analyst"     # which specialist gets the Module 2 paper-agent tools
CUSTOM_AGENTS = {}                             # see the ⭐ stretch task

PLAN = vbt_plan(TEAM, CUSTOM_AGENTS, PAPER_AGENT_HOLDER)
if PLAN:
    if PLAN["unknown"]:
        print("⚠️ Not a Virtual Biotech agent, so left out: " + ", ".join(PLAN["unknown"]))
    table(["Agent", "Model", "Effort", "Data tools (by server)", "Other tools"],
          [[a["name"], a["model"], a.get("effort") or "default",
            "; ".join(f"{s} ({len(t)})" for s, t in a["tools_by_server"].items()) or "—",
            ", ".join(t for t in a["builtin_tools"] if t in ("WebSearch", "WebFetch", "Bash")) or "—"]
           for a in PLAN["team"]], title="Your team (the CSO is always included)")
    pa = paper_agents()
    attached = [s for s in (pa.get("servers") or {}) if any(s in a["tools_by_server"] for a in PLAN["team"])]
    print(f"Paper agents attached to {PAPER_AGENT_HOLDER}: {', '.join(attached)}" if attached else
          "No Module 2 paper agents attached (no paper_agents.json found, or the holder isn't on the team).")

In [ ]:
# Routing trace for *your* team. Watch for analyses that nobody can take.
if PLAN:
    PREDICTED = routing_preview(PLAN, ANALYSES)

### 2b · The brief
This is the message the CSO receives. It's built from your Module 1 strategy and your Module 2 analysis log, and it labels earlier conclusions as **hypotheses to test**. You can add instructions at the end.

In [ ]:
EXTRA_INSTRUCTIONS = ""   # e.g. "Pay particular attention to expression in vascular endothelium."

def build_brief():
    lines = [f"# Research brief: {GENE} in {DISEASE}", "",
             strategy.get("question") or f"Is {GENE} a credible therapeutic target in {DISEASE}, and which modality fits it?", "",
             "## What earlier work suggests (hypotheses to test, not facts)"]
    if strategy:
        lines += [f"- Hypothesis: {strategy.get('hypothesis', 'not specified')}",
                  f"- An AI team meeting working from memory, without data, recommended: {strategy.get('recommended_modality', 'not specified')}"]
        lines += [f"- Safety concern to check: {s}" for s in strategy.get("safety_checks", [])]
        lines += [f"- We would abandon the target if: {s}" for s in strategy.get("kill_criteria", [])]
        notes = (strategy.get("provenance") or {}).get("human_review", {}).get("notes")
        if notes:
            lines.append(f"- Our own note: {notes}")
    else:
        lines.append("- None yet.")
    lines += ["", "## Analyses we planned"]
    lines += [f"- {a.get('id')} ({a.get('tool')}): {a.get('question')}"
              + (f" Counts against the target if: {a['evidence_against']}" if a.get("evidence_against") else "") for a in ANALYSES]
    lines += ["", "## Results already in hand from paper agents (Module 2)"]
    log = (_read_json(MODULE2_DIR / "analysis_log.json")
           or _read_json(WORKSHOP_DIR / "cache" / "module2" / "analysis_log.json") or [])
    verdict = {True: "our re-run agrees", False: "our re-run DISAGREES", None: "not checked by us"}
    lines += [f"- {e.get('analysis_id')} · {e.get('server')}.{e.get('tool')}: {e.get('claude_said') or '(no summary)'} "
              f"({verdict.get(e.get('human_check'), str(e.get('human_check')))}){' Note: ' + e['notes'] if e.get('notes') else ''}"
              for e in log] or ["- None yet."]
    lines += ["", "## What we need from you",
              f"Assess whether {GENE} is a credible therapeutic target in {DISEASE}, and which modality fits it, using your "
              "specialists' data and tools. Test the hypotheses above rather than assuming them, and look for evidence "
              "against the target as well as for it.",
              "Structure the final answer under these headings: Target rationale; Safety liabilities; Suggested modality; "
              "Open questions and next steps.",
              "File every factual statement as a claim with evidence. Where data was unavailable or a tool failed, say so."]
    if EXTRA_INSTRUCTIONS:
        lines += ["", EXTRA_INSTRUCTIONS]
    return redact("\n".join(lines))

BRIEF = build_brief()
(OUTPUT_DIR / "brief.md").write_text(BRIEF)
display(Markdown(BRIEF))

### 2c · Run the team
This starts the real organisation, on your Anthropic key. A progress line appears every 20 seconds or so. After **`TIME_LIMIT_MIN`** minutes the team's research is stopped. The CSO then gets **`WRAPUP_MIN`** more minutes to write the report and file claims from what is already on record, so even a cut-short run ends with something you can check.

While it runs, open the run folder in VS Code's file explorer and watch `work/` fill up, one folder per specialist.

To avoid paying twice by accident, this cell won't start a second run unless you set `RERUN = True`.

In [ ]:
RERUN = False
last_run_file = OUTPUT_DIR / "last_run.json"

if USE_CACHE:
    print("USE_CACHE is on: skipping the live run. Go on to Section 3.")
elif last_run_file.exists() and not RERUN:
    print(f"You already have a run ({_read_json(last_run_file).get('run_dir')}). Set RERUN = True to start another.")
elif not os.environ.get("ANTHROPIC_API_KEY"):
    print("❌ No Anthropic key. Run Step 0 of 00_preflight.ipynb, or set USE_CACHE = True.")
else:
    print(f"Starting the Virtual Biotech · team of {len(PLAN['team']) if PLAN else '?'} + CSO · "
          f"limit {TIME_LIMIT_MIN} min · expected cost {EXPECTED_COST}\n")
    (OUTPUT_DIR / "brief.md").write_text(BRIEF)
    cfg = write_config(TEAM, CUSTOM_AGENTS, PAPER_AGENT_HOLDER)
    events, code, tail = launch([LAUNCHER, "run", "--config", cfg, "--query-file", OUTPUT_DIR / "brief.md"],
                                with_key=True, timeout_s=(TIME_LIMIT_MIN + WRAPUP_MIN) * 60 + 600, on_event=show_event)
    result = next((e for e in reversed(events) if e.get("kind") == "result"), None)
    if result is None:   # force-stopped: no summary, but the run folder may still hold work worth reading
        folder = next((e.get("path") for e in events if e.get("kind") == "run_dir"), None)
        manifest = _read_json(Path(folder) / "MANIFEST.json") if folder else None
        result = {"run_dir": folder, "status": (manifest or {}).get("status", "stopped"), "turn_status": None}
    run_dir = Path(result["run_dir"]) if result.get("run_dir") else None
    did_work = run_dir is not None and (
        result.get("status") not in (None, "unknown", "in_progress", "no run folder") or result.get("turn_status")
        or (run_dir / "report" / "FINAL_REPORT.md").exists()
        or ((run_dir / "logs" / "trace.jsonl").exists() and (run_dir / "logs" / "trace.jsonl").stat().st_size > 0))
    if did_work:
        shutil.copy(cfg, run_dir / "workshop_settings.json")   # the run's own manifest records the repo's defaults
        last_run_file.write_text(json.dumps({**result, "team": TEAM, "custom_agents": list(CUSTOM_AGENTS),
                                             "model": MODEL, "effort": EFFORT, "brief": str(OUTPUT_DIR / "brief.md")}, indent=2))
        stopped = result.get("stopped_by_time_limit") or result.get("stopped_by_user") or result.get("status") != "completed"
        cost = cost_text(run_dir) + " (see 3d)"
        if result.get("stopped_by_time_limit"):
            print("\n📝 The research was cut off at the time limit; "
                  + ("the CSO wrote up what was on record." if result.get("wrapped_up") else "the write-up didn't finish either."))
        print(f"\n{'✅' if result.get('status') == 'completed' else '⚠️'} Run {result.get('status')} in "
              f"{result.get('elapsed_s', 0) // 60} min · cost {cost} · {result.get('n_claims', 0)} claims filed · "
              f"specialists: {', '.join(result.get('agents') or []) or 'none'}")
    else:
        print("\n❌ The team didn't get started, so there's nothing to read. Last lines of the log:\n  "
              + "\n  ".join(tail[-15:]))
        print("\nShow this to a helper. You can retry by running this cell again.")
        if (CACHE_DIR / "last_run.json").exists():
            print("Or continue with the pre-run copy: set USE_CACHE = True and run Section 3.")

## 3 · Read the report · 16:20–16:30

In [ ]:
source = (CACHE_DIR if USE_CACHE else OUTPUT_DIR) / "last_run.json"
RUN = _read_json(source) or {}
RUN_DIR = Path(RUN["run_dir"]) if RUN.get("run_dir") else None
if USE_CACHE and RUN_DIR and not RUN_DIR.exists():
    RUN_DIR = CACHE_DIR / "runs" / RUN_DIR.name     # the pre-run copy lives in the cache
if RUN_DIR and RUN_DIR.exists():
    print(f"{'📦 Pre-run copy' if USE_CACHE else 'Your run'}: {RUN_DIR}")
    print(f"Status: {RUN.get('status')} · audit page: file://{RUN_DIR / 'audit.html'} (open it in Firefox)")
else:
    print("No run to read yet. Run 2c, or set USE_CACHE = True.")

### 3a · The answer
Read it as a sceptical reviewer. The quick view below pulls out the three things you'll be asked about: **rationale, safety and modality**.

In [ ]:
REPORT = ""
if RUN_DIR:
    for rel in ("report/FINAL_REPORT.md", "README.md"):
        if (RUN_DIR / rel).exists():
            REPORT = (RUN_DIR / rel).read_text()
            break
    shown = re.sub(r"\[\[claim:([A-Za-z0-9_.-]+)\]\]", r" `[\1]`", REPORT)
    for heading in ("rationale", "safety", "modality"):
        part = section(shown, heading)
        print(f"── {heading.upper()} " + "─" * 50)
        print(redact(part[:1500]) + (" …" if part and len(part) > 1500 else "") if part else
              f"(no heading containing '{heading}', so read the full report below)")
    show_text("The full report (click to expand)", shown)

### 3b · Every claim and its source
The `[C1]`-style markers in the report point to these claims. For each one, check the evidence:
- **✓ verified:** a file or tool call that's on record in this run.
- **external:** a paper or URL the run couldn't check. Someone has to read it.
- **not on record:** a defect. The claim points to something that doesn't exist.

In [ ]:
CLAIMS = []
if RUN_DIR:
    data = _read_json(RUN_DIR / "evidence" / "claims.json") or {}
    CLAIMS = data.get("claims", []) if isinstance(data, dict) else data

def _status(ev):
    if ev.get("verified"):
        return "✓ verified"
    return "external" if ev.get("kind") == "citation" else "not on record"

def _what(ev):
    for key in ("path", "tool_name", "pmid", "doi", "url", "tool_use_id"):
        if ev.get(key):
            who = ev.get("produced_by") or ev.get("agent")
            return f"{ev.get('kind')}: {ev[key]}" + (f" (by {who})" if who else "")
    return ev.get("kind", "?")

rows, counts = [], collections.Counter()
for c in CLAIMS:
    evidence = c.get("evidence") or []
    for ev in evidence:
        counts[_status(ev)] += 1
    rows.append([c.get("id"), c.get("text"), c.get("confidence"),
                 "\n".join(f"{_status(ev)} · {_what(ev)}" for ev in evidence) or "none"])
if CLAIMS:
    table(["ID", "Claim", "Confidence", "Evidence"], rows, title=f"{len(CLAIMS)} claims")
    only_external = [c.get("id") for c in CLAIMS if c.get("evidence") and all(_status(ev) == "external" for ev in c["evidence"])]
    print("Evidence entries: " + ", ".join(f"{n} {k}" for k, n in counts.most_common()))
    if only_external:
        print(f"📚 Supported only by external references nobody has checked yet: {', '.join(only_external)}. Module 4 starts here.")
    if counts.get("not on record"):
        print("❌ Some evidence isn't on record: treat those claims as unsupported.")
    refs = set(re.findall(r"\[\[claim:([A-Za-z0-9_.-]+)\]\]", REPORT))
    print(f"The report cites {len(refs)} of the {len(CLAIMS)} filed claims inline.")
elif RUN_DIR:
    print("No claims were filed. If the run was interrupted, the CSO may not have reached that step; "
          "otherwise, every statement in the report is unsupported.")

### 3c · Who did what, and what failed
This is the actual routing, from the run's trace. Compare it with your dry-run prediction. Failed tool calls matter: a failed query is **no evidence at all**, not evidence of absence.

In [ ]:
TRACE = []
if RUN_DIR:
    p = RUN_DIR / "logs" / "trace.jsonl"
    if not p.exists():
        p = RUN_DIR / "trace.jsonl"
    if p.exists():
        for line in p.read_text().splitlines():
            try:
                TRACE.append(json.loads(line))
            except ValueError:
                pass

if TRACE:
    t0 = datetime.datetime.fromisoformat(TRACE[0]["ts"])
    def at(e):
        return (datetime.datetime.fromisoformat(e["ts"]) - t0).total_seconds()
    calls = [e for e in TRACE if e.get("type") in ("tool_end", "tool_error")]
    per_agent = collections.Counter(e.get("agent") or "CSO" for e in calls)
    stops = {e.get("agent_id"): e for e in TRACE if e.get("type") == "agent_stop"}
    rows = []
    for e in (e for e in TRACE if e.get("type") == "agent_start"):
        stop = stops.get(e.get("agent_id"), {})
        dur = stop.get("duration_s")
        rows.append([e.get("agent_type"), f"{at(e) / 60:.1f} min", f"{dur / 60:.1f} min" if dur else "did not finish",
                     per_agent.get(e.get("agent_type"), 0)])
    table(["Agent", "Started at", "Took", "Tool calls"], rows, title="Timeline")
    ACTUAL = {e.get("agent_type") for e in TRACE if e.get("type") == "agent_start"}
    predicted = set(globals().get("PREDICTED", set()))
    print(f"Predicted and used: {', '.join(sorted(predicted & ACTUAL)) or '—'}")
    print(f"Predicted but never called: {', '.join(sorted(predicted - ACTUAL)) or '—'}")
    print(f"Called without being predicted: {', '.join(sorted(ACTUAL - predicted)) or '—'}\n")
    failed = [e for e in calls if e.get("type") == "tool_error" or e.get("is_error")]
    if failed:
        table(["Tool", "Agent", "Error"], [[e.get("tool_name"), e.get("agent") or "CSO",
                                             str(e.get("error") or e.get("tool_response"))[:200]] for e in failed],
              title=f"❌ {len(failed)} failed tool calls: does the report admit to each one?")
    else:
        print(f"✅ No failed tool calls among {len(calls)}.")
elif RUN_DIR:
    print("No trace found in this run folder.")

### 3d · Cost, and an integrity check

In [ ]:
if RUN_DIR:
    cost = _read_json(RUN_DIR / "logs" / "cost_report.json") or {}
    low, high = run_cost(RUN_DIR)
    per = [(e.get("agent_type"), (e.get("cost") or {}).get("total_cost_usd") or (e.get("cost") or {}).get("cost_usd"))
           for e in TRACE if e.get("type") == "agent_stop"]
    per = [(a, c) for a, c in per if isinstance(c, (int, float))]
    shown = cost_text(RUN_DIR)
    if high and abs(high - low) >= 0.01:
        shown += (f" (the run's own total is US${cost.get('total_cost_usd') or 0:.2f}; the specialists reported "
                  f"US${sum(c for _, c in per):.2f}, and not all of that is included in the run's total)")
    if RUN.get("stopped_by_time_limit") or RUN.get("stopped_by_user"):
        shown += ". The run was cut short, and agents that didn't finish report nothing, so it may have cost more"
    print(f"Total cost: {shown} · turns: {cost.get('num_turns', '?')} · trace events: {cost.get('trace_events', len(TRACE))}")
    if per:
        table(["Agent", "Cost (US$)"], [[a, f"{c:.2f}"] for a, c in sorted(per, key=lambda x: -x[1])], title="Cost by agent")

    # Did your key end up in the run's records? (Agents' shells can see it; the guards reduce, but can't remove, that risk.)
    key = os.environ.get("ANTHROPIC_API_KEY")
    exposed = []
    for f in RUN_DIR.rglob("*"):
        try:
            if f.is_file() and f.stat().st_size < 200_000_000 and f.suffix.lower() not in (".png", ".jpg", ".h5ad", ".parquet", ".pdf"):
                text = f.read_text(errors="ignore")
                if (key and key in text) or _SECRET_RE.search(text):
                    exposed.append(str(f.relative_to(RUN_DIR)))
        except OSError:
            continue
    if exposed:
        print(f"\n⚠️ Something key-shaped appears in {len(exposed)} file(s) of this run (e.g. {exposed[0]}). "
              "Tell a helper: don't share this run folder, and the key should be revoked after the workshop.")
    else:
        print("\n✅ No key appears anywhere in this run's records.")

    # The Virtual Biotech's own verifier: re-hashes every artifact and re-resolves every claim. No model calls.
    proc = subprocess.run(_command([VBT_DIR / "run_vbt.py", "verify", RUN_DIR, "--json"]), cwd=str(VBT_DIR),
                          env=_child_env(False), capture_output=True, text=True, timeout=300, stdin=subprocess.DEVNULL)
    try:
        report = json.loads(proc.stdout[proc.stdout.find("{"):])
        print(f"\nVerification: {report.get('status', '?')} ({'ok' if report.get('ok') else 'problems found'})")
        for p in (report.get("problems") or [])[:10]:
            print("  • " + redact(json.dumps(p, default=str))[:250])
        print("Verification checks the record (files unchanged, claims linked). It does NOT check that the science is right.")
    except ValueError:
        print("Couldn't run the verifier:\n" + redact((proc.stderr or proc.stdout)[-1000:]))

In [ ]:
# Hand-off to Module 4.
if RUN_DIR:
    summary = {**RUN, "run_dir": str(RUN_DIR), "from_cache": USE_CACHE, "n_claims": len(CLAIMS),
               "claim_evidence": dict(counts) if CLAIMS else {}, "agents_called": sorted(globals().get("ACTUAL", [])),
               "predicted_agents": sorted(globals().get("PREDICTED", [])), "saved": datetime.datetime.now().isoformat(timespec="seconds")}
    (OUTPUT_DIR / "run_summary.json").write_text(json.dumps(summary, indent=2, default=str))
    print(f"✅ Saved {OUTPUT_DIR / 'run_summary.json'}. Module 4 picks up from here.")

## ⭐ Stretch: add a new division agent
A division agent is three things: a **description** (what the CSO reads when routing), a **system prompt**, and a **tool list**. Here's an immuno-oncology analyst, a gap in the default organisation for a question about an immune checkpoint molecule. Run the dry run to see whether the CSO's likely routing changes, then set `RERUN = True` in 2c to try it for real.

Tool names follow `mcp__<server>__<tool>`; the table in 2a shows what each server offers.

In [ ]:
CUSTOM_AGENTS = {
    "immuno-oncology-analyst": {
        "description": "[Immuno-oncology] Tumour immune microenvironment: expression of a target on tumour, immune "
                       "and stromal cells; checkpoint biology; what that means for antibody, ADC and cell-therapy modalities.",
        "prompt": ("You are an immuno-oncology scientist in a therapeutic research organisation. Assess how the target "
                   "is expressed on malignant, immune and stromal cells, and what its immune-checkpoint biology implies "
                   "for the candidate modalities. Use only your tools, save every result in your workspace, register "
                   "the files, and report the tool-use IDs behind each finding. Say plainly when data is missing."),
        "tools": ["mcp__single_cell__search_genes", "mcp__single_cell__count_cells",
                  "mcp__single_cell__query_cell_metadata", "mcp__single_cell__get_anndata",
                  "mcp__expression__query_expression_by_gene", "mcp__target__get_target_info",
                  "mcp__pathway__get_gene_pathways", "mcp__interaction__get_interactions"],
        "model": "inherit",
    },
}
PLAN = vbt_plan(TEAM, CUSTOM_AGENTS, PAPER_AGENT_HOLDER)
if PLAN:
    PREDICTED = routing_preview(PLAN, ANALYSES)

---
### For organisers
- **Set up the Virtual Biotech on your instance before the image is made.** Follow the facilitators' setup guide: its own conda environment (`vbt`, Python 3.11), the Open Targets 25.09 download (29 GB), `python setup_mcp.py`, and `python tools/doctor.py --skip-api-key --smoke`. **Don't create the repo's `.env` with a key in it.** This notebook passes the key from the private key file to the run process only.
- **Disk:** the Virtual Biotech needs about 30 GB for Open Targets alone, so the instances' 30 GB disks are too small. Ask RAIL for at least 100 GB, or a shared read-only volume holding the Open Targets data.
- **Build the pre-run copy:** with the default team, run 2c once. Then copy `outputs/module3/last_run.json` to `cache/module3/` and the run folder to `cache/module3/runs/`. `prepare_image.py` refuses to clear your outputs until that's done.
- **Time limit and write-up:** after `TIME_LIMIT_MIN` the launcher stops the research and gives the CSO `WRAPUP_MIN` more minutes, in the same session, to write the report and file claims from what is on record. Specialists are told to report back within about half the limit and to avoid large downloads, and the CSO to call each specialist once. Without these, a test run's single-cell analyst spent 11 minutes downloading Census data and the run ended with no report and no claims.
- **Cost and load:** each run is a CSO plus five agents working in parallel, with many tool calls. Measured on 18 November 2026 with the default team: a run that finished in 12 minutes cost US$3.78–6.30 (26 claims, all verified), and a run cut off at the time limit cost at least US$7.40. The run's own total doesn't include specialists that run in the background, so the notebook shows a range. With 30+ participants starting at once, the load on your Anthropic organisation is real: run in pairs (one run per pair) and check your rate limits.
- **Model:** `MODEL` must be available to the workshop keys; the self-test checks this if its `VBT_MODEL` setting matches. To match the published runs, use the repo's default (`claude-sonnet-4-5-20250929`) and set `EFFORT = "high"` and `CSO_EFFORT = "xhigh"`, if that model is still offered; expect slower, costlier runs.